# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring / Ranking**

This is a ranking problem, not a classification problem. The goal is not to label pages as "good" or "bad" but to produce a continuous opportunity score that ranks pages from most urgent to least urgent for content review.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (ideal, unmeasurable):**
Whether a page is genuinely under-capturing clicks relative to its true potential given its position and content quality.

**Proxy (measurable):**
`ctr_gap` : the difference between a page's actual CTR and the average CTR for pages in the same position tier.

A strongly negative `ctr_gap` suggests the page is getting fewer clicks when compared to pages of the same position.

**Proxy limitations:**
- Low CTR may reflect SERP feature competition, not title quality
- Low CTR may reflect keyword intent mismatch
- The proxy assumes position tier is the right grouping variable


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: Precision@K (K = 20, K = 50)**

Of the top-K pages ranked by opportunity score, what fraction carry the `is_declining_label = 1` flag? A higher fraction is better.

**Baseline to beat:**
A random ranking achieves Precision@K equal to the base rate of declining pages in the dataset. The scoring model must exceed this to justify its use.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent
data_path = project_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


One content page (one `content_id`) with aggregated 90-day search and engagement metrics.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like flagging all pages with CTR < 2%) fails because:

1. CTR varies naturally by position. Position 1 pages always have higher CTR than position 8 pages. A flat threshold treats them identically without factoring in positions.

2. A rule produces a binary flag, not a ranked list for prioritization. Content teams need to see which pages to prioritize, not just a binary label.

3. A rule cannot weight by impression volume like an ML model. A page losing clicks at 50,000 impressions is more urgent than the same CTR gap at 500 impressions.

The scoring approach handles all three by computing position-adjusted gaps and combining multiple signals into one weighted score, which is more efficient than fixed hand rules.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.